In [1]:
import requests
import pandas as pd
from owslib.wms import WebMapService
from owslib.wfs import WebFeatureService
import geopandas as gpd
from shapely.geometry import Point
import io

# URL do usługi WMS dla RCN
wms_url = "https://mapy.geoportal.gov.pl/wss/service/rcn"

# Łączenie z WMS za pomocą owslib
wms = WebMapService(wms_url, version='1.3.0')

print("Dostępne warstwy WMS:")
for layer_name in wms.contents:
    print(layer_name)

# Przykład GetFeatureInfo dla warstwy 'budynki' (jeśli queryable) - zmień format na 'text/xml'
layer = 'budynki'
if layer in wms.contents and wms[layer].queryable:
    # Współrzędne dla Lublina (EPSG:2180) - bbox wyznaczony z geoportalu
    bbox = (741742.24, 373830.30, 754350.03, 387807.82)  # xmin, ymin, xmax, ymax
    size = (101, 101)
    pos = (50, 50)  # pozycja w pikselach

    # Zmiana formatu na 'text/xml' (z Capabilities)
    info = wms.getfeatureinfo(layers=[layer], srs='EPSG:2180', bbox=bbox, size=size, format='text/xml', query_layers=[layer], xy=pos)
    print("Informacje z GetFeatureInfo (XML):")
    print(info.read().decode('utf-8'))
else:
    print(f"Warstwa {layer} nie jest queryable lub nie istnieje.")

# Próba WFS (jeśli dostępny) - zwiększ timeout
wfs_url = wms_url.replace('WMS', 'WFS')
try:
    wfs = WebFeatureService(wfs_url, version='1.1.0', timeout=60)  # Zwiększony timeout
    print("Dostępne warstwy WFS:")
    for feature_type in wfs.contents:
        print(feature_type)
    
    # Pobierz dane dla jednej warstwy (np. budynki) w GML - ogranicz liczbę features
    if 'budynki' in wfs.contents:
        response = wfs.getfeature(typename='budynki', outputFormat='text/xml; subtype=gml/3.1.1', maxfeatures=20)  # Ogranicz do 10 features
        # Zapisz do pliku tymczasowego lub sparsuj
        with open('temp.gml', 'wb') as f:
            f.write(response.read())
        try:
            gdf = gpd.read_file('temp.gml', driver='GML')
            print("Dane WFS pobrane (GML):")
            print(gdf.head())
        except Exception as e:
            print(f"Błąd parsowania GML: {e}")
            # Alternatywnie, wyświetl surową odpowiedź
            response = wfs.getfeature(typename='budynki', outputFormat='text/xml; subtype=gml/3.1.1', maxfeatures=10)
            print("Surowa odpowiedź GML (pierwsze 1000 znaków):")
            print(response.read().decode('utf-8')[:1000])
except Exception as e:
    print(f"WFS niedostępny: {e}")

# Uwaga: Pełne dane tabelaryczne RCN wymagają wniosku elektronicznego.

Dostępne warstwy WMS:
budynki
lokale
dzialki
powiaty
Informacje z GetFeatureInfo (XML):
<?xml version='1.0' encoding="UTF-8" standalone="no" ?>
<ServiceExceptionReport version="1.3.0" xmlns="http://www.opengis.net/ogc" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengis.net/ogc http://schemas.opengis.net/wms/1.3.0/exceptions_1_3_0.xsd">
<ServiceException code="InvalidFormat">
msWMSLoadGetMapParams(): Image handling error. Unsupported output format (text/xml).
</ServiceException>
</ServiceExceptionReport>

Informacje z GetFeatureInfo (XML):
<?xml version='1.0' encoding="UTF-8" standalone="no" ?>
<ServiceExceptionReport version="1.3.0" xmlns="http://www.opengis.net/ogc" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengis.net/ogc http://schemas.opengis.net/wms/1.3.0/exceptions_1_3_0.xsd">
<ServiceException code="InvalidFormat">
msWMSLoadGetMapParams(): Image handling error. Unsupported output format (text/xm

/home/dron/Studia/eskploracja/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GML does not support open option DRIVER
  return ogr_read(


In [3]:
# Analiza danych z WFS
try:
    # Załaduj dane z pliku temp.gml, jeśli istnieje
    gdf = gpd.read_file('temp.gml', driver='GML')
    print("Dane załadowane z pliku GML.")
    print(f"Liczba rekordów: {len(gdf)}")
    print("Kolumny:", list(gdf.columns))
    
    # Filtruj dane dla Lublina (np. po adresie zawierającym 'Lublin' lub kodzie TERYT)
    # TERYT dla Lublina: powiat 0662, gmina 066201
    lublin_data = gdf[gdf['bud_adres'].str.contains('Lublin', na=False, case=False) | (gdf['teryt'].str.startswith('0662', na=False))]
    print(f"Dane dla Lublina: {len(lublin_data)} rekordów")
    
    if not lublin_data.empty:
        # Statystyki cen (nier_cena_brutto)
        ceny = pd.to_numeric(lublin_data['nier_cena_brutto'], errors='coerce').dropna()
        if not ceny.empty:
            print("Podstawowe statystyki cen sprzedaży w Lublinie:")
            print(ceny.describe())
            
            # Wykres cen
            import matplotlib.pyplot as plt
            plt.hist(ceny, bins=50)
            plt.title('Rozkład cen nieruchomości w Lublinie')
            plt.xlabel('Cena brutto')
            plt.ylabel('Liczba transakcji')
            plt.show()
        else:
            print("Brak danych cenowych do analizy.")
    else:
        print("Brak danych dla Lublina.")
except Exception as e:
    print(f"Błąd ładowania danych: {e}")

Dane załadowane z pliku GML.
Liczba rekordów: 20
Kolumny: ['gml_id', 'serwis_rcn', 'teryt', 'tran_przestrzen_nazw', 'tran_lokalny_id_iip', 'tran_wersja_id', 'tran_rodzaj_trans', 'tran_rodzaj_rynku', 'tran_sprzedajacy', 'tran_kupujacy', 'tran_cena_brutto', 'tran_vat', 'dok_data', 'nier_rodzaj', 'nier_prawo', 'nier_udzial', 'nier_pow_gruntu', 'nier_cena_brutto', 'nier_vat', 'bud_id_budynku', 'bud_nr_budynku', 'bud_rodzaj', 'bud_pow_uzyt', 'bud_cena_brutto', 'bud_vat', 'bud_adres', 'geometry']
Dane dla Lublina: 0 rekordów
Brak danych dla Lublina.


/home/dron/Studia/eskploracja/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GML does not support open option DRIVER
  return ogr_read(


In [8]:
gdf.describe()

,gml_id,serwis_rcn,teryt,tran_przestrzen_nazw,tran_lokalny_id_iip,tran_wersja_id,tran_rodzaj_trans,tran_rodzaj_rynku,tran_sprzedajacy,tran_kupujacy,...,nier_cena_brutto,nier_vat,bud_id_budynku,bud_nr_budynku,bud_rodzaj,bud_pow_uzyt,bud_cena_brutto,bud_vat,bud_adres,geometry
count,20,0,20,19,20,19,20,14,20,20,...,9,3,20,20,19,2,0,0,16,20
unique,20,0,7,6,20,18,1,2,2,1,...,9,3,19,19,3,2,0,0,15,19
top,budynki.26880499,NaN,2208,PL.PZGiK.5715.RCiWN,1F4E8E3D-1477-41AB-8562-C1ACC18E23A6,2011-12-16T19:58:04,wolnyRynek,pierwotny,osobaPrawna,osobaFizyczna,...,970000,44187.78,220801_1.0003.1141_BUD,1141_BUD,mieszkalny,794.32,NaN,NaN,MSC:Lębork;UL:Kardynała Stefana Wyszyńskiego;N...,"POLYGON ((743025.669856 419111.929317, 743020...."
freq,1,NaN,7,7,1,2,20,10,13,20,...,1,1,2,2,17,1,NaN,NaN,2,2


In [7]:
gdf.columns

Index(['gml_id', 'serwis_rcn', 'teryt', 'tran_przestrzen_nazw',
       'tran_lokalny_id_iip', 'tran_wersja_id', 'tran_rodzaj_trans',
       'tran_rodzaj_rynku', 'tran_sprzedajacy', 'tran_kupujacy',
       'tran_cena_brutto', 'tran_vat', 'dok_data', 'nier_rodzaj', 'nier_prawo',
       'nier_udzial', 'nier_pow_gruntu', 'nier_cena_brutto', 'nier_vat',
       'bud_id_budynku', 'bud_nr_budynku', 'bud_rodzaj', 'bud_pow_uzyt',
       'bud_cena_brutto', 'bud_vat', 'bud_adres', 'geometry'],
      dtype='str')

In [6]:
gdf.head()

,gml_id,serwis_rcn,teryt,tran_przestrzen_nazw,tran_lokalny_id_iip,tran_wersja_id,tran_rodzaj_trans,tran_rodzaj_rynku,tran_sprzedajacy,tran_kupujacy,...,nier_cena_brutto,nier_vat,bud_id_budynku,bud_nr_budynku,bud_rodzaj,bud_pow_uzyt,bud_cena_brutto,bud_vat,bud_adres,geometry
0,budynki.26880499,None,2461,PL.PZGiK.89.RCN,1F4E8E3D-1477-41AB-8562-C1ACC18E23A6,2023-09-22T03:09:47,wolnyRynek,pierwotny,osobaFizyczna,osobaFizyczna,...,NaN,NaN,246101_1.0009.3845_BUD,3845_BUD,NaN,NaN,None,None,MSC:Bielsko-Biała;UL:Jana Kochanowskiego,"POLYGON ((215659.899 502381.342, 215660.662 50..."
1,budynki.24937987,None,2210,PL.PZGiK.253.RCiWN,cb1101f6-c85b-44f2-9c14-88660ccbe97f,2011-12-16T19:58:04,wolnyRynek,NaN,osobaPrawna,osobaFizyczna,...,NaN,NaN,221002_4.0002.398_BUD,398_BUD,mieszkalny,NaN,None,None,MSC:Nowy Dwór Gdański;UL:Sikorskiego;NR_PORZ:2,"POLYGON ((705357.435 507655.963, 705356.587 50..."
2,budynki.24938071,None,2210,PL.PZGiK.253.RCiWN,2c7e8bab-3fc8-4efb-b283-aee57ef12053,2011-12-16T19:58:04,wolnyRynek,NaN,osobaPrawna,osobaFizyczna,...,NaN,NaN,221005_2.0002.1680_BUD,1680_BUD,mieszkalny,NaN,None,None,NaN,"POLYGON ((719895.567 515362.716, 719890.096 51..."
3,budynki.24459670,None,0463,PL.PZGiK.7601.RCiWN,6315ff6d-0477-4645-958d-d0d0273bded2,2025-10-14T09:26:18,wolnyRynek,wtorny,osobaPrawna,osobaFizyczna,...,970000,NaN,046301_1.0017.40_BUD,40_BUD,mieszkalny,794.32,None,None,MSC:Toruń;UL:Rynek Nowomiejski;NR_PORZ:5,"POLYGON ((571837.127 473942.401, 571826.063 47..."
4,budynki.24463304,None,0463,PL.PZGiK.7601.RCiWN,2418f3da-e669-4687-a7da-aa4685295550,2025-12-17T09:26:32,wolnyRynek,pierwotny,osobaPrawna,osobaFizyczna,...,603000,NaN,046301_1.0030.222_BUD,222_BUD,mieszkalny,NaN,None,None,MSC:Toruń;UL:Jana Michała Hubego;NR_PORZ:30,"POLYGON ((575764.854 473667.081, 575748.391 47..."


In [10]:
# Wyświetlanie pełnego DataFrame bez chowania kolumn
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_rows', 100)
# Teraz wyświetlenie gdf pokaże wszystkie kolumny
gdf.head()

,gml_id,serwis_rcn,teryt,tran_przestrzen_nazw,tran_lokalny_id_iip,tran_wersja_id,tran_rodzaj_trans,tran_rodzaj_rynku,tran_sprzedajacy,tran_kupujacy,tran_cena_brutto,tran_vat,dok_data,nier_rodzaj,nier_prawo,nier_udzial,nier_pow_gruntu,nier_cena_brutto,nier_vat,bud_id_budynku,bud_nr_budynku,bud_rodzaj,bud_pow_uzyt,bud_cena_brutto,bud_vat,bud_adres,geometry
0,budynki.26880499,None,2461,PL.PZGiK.89.RCN,1F4E8E3D-1477-41AB-8562-C1ACC18E23A6,2023-09-22T03:09:47,wolnyRynek,pierwotny,osobaFizyczna,osobaFizyczna,177490,8,2011-12-21 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,NaN,NaN,246101_1.0009.3845_BUD,3845_BUD,NaN,NaN,None,None,MSC:Bielsko-Biała;UL:Jana Kochanowskiego,"POLYGON ((215659.899 502381.342, 215660.662 50..."
1,budynki.24937987,None,2210,PL.PZGiK.253.RCiWN,cb1101f6-c85b-44f2-9c14-88660ccbe97f,2011-12-16T19:58:04,wolnyRynek,NaN,osobaPrawna,osobaFizyczna,160000,NaN,2009-01-05 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,NaN,42,NaN,NaN,221002_4.0002.398_BUD,398_BUD,mieszkalny,NaN,None,None,MSC:Nowy Dwór Gdański;UL:Sikorskiego;NR_PORZ:2,"POLYGON ((705357.435 507655.963, 705356.587 50..."
2,budynki.24938071,None,2210,PL.PZGiK.253.RCiWN,2c7e8bab-3fc8-4efb-b283-aee57ef12053,2011-12-16T19:58:04,wolnyRynek,NaN,osobaPrawna,osobaFizyczna,145443.44,NaN,2010-11-26 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,NaN,17,NaN,NaN,221005_2.0002.1680_BUD,1680_BUD,mieszkalny,NaN,None,None,NaN,"POLYGON ((719895.567 515362.716, 719890.096 51..."
3,budynki.24459670,None,0463,PL.PZGiK.7601.RCiWN,6315ff6d-0477-4645-958d-d0d0273bded2,2025-10-14T09:26:18,wolnyRynek,wtorny,osobaPrawna,osobaFizyczna,970000,NaN,2025-09-24 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,970000,NaN,046301_1.0017.40_BUD,40_BUD,mieszkalny,794.32,None,None,MSC:Toruń;UL:Rynek Nowomiejski;NR_PORZ:5,"POLYGON ((571837.127 473942.401, 571826.063 47..."
4,budynki.24463304,None,0463,PL.PZGiK.7601.RCiWN,2418f3da-e669-4687-a7da-aa4685295550,2025-12-17T09:26:32,wolnyRynek,pierwotny,osobaPrawna,osobaFizyczna,603000,NaN,2025-07-23 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,NaN,603000,NaN,046301_1.0030.222_BUD,222_BUD,mieszkalny,NaN,None,None,MSC:Toruń;UL:Jana Michała Hubego;NR_PORZ:30,"POLYGON ((575764.854 473667.081, 575748.391 47..."


# Pobieranie danych z Rejestru Cen Nieruchomości (RCN)

Ten notebook pokazuje, jak wyciągnąć dane dotyczące cen nieruchomości z geoportalu przy użyciu Pythona.  
**Uwaga:** Rejestr Cen Nieruchomości (RCN) może wymagać uwierzytelnienia lub dostępu przez oficjalne kanały. Sprawdź dokumentację geoportalu dla dokładnych endpointów API lub linków do plików danych.